# 🖊️ DisgraPhi: Personalize Your Handwriting OCR

Welcome to DisgraPhi! This notebook will help you create a personalized handwriting recognition model trained specifically on **your** handwriting.

## What This Does

- Trains a vision-language model to recognize YOUR handwriting
- Works great for dysgraphia, motor difficulties, or just unique penmanship
- Takes ~15-30 minutes to complete
- Requires only a few samples of your handwriting

## Before You Start

You'll need:
1. **Handwriting samples**: 10-30 photos/scans of your handwriting
2. **Ground truth text**: What each sample says (we'll help you create this)
3. **GPU runtime**: Enable GPU in Runtime → Change runtime type → T4 GPU

**💡 Pro Tip**: For best results, use our [Gradio Data Collector](https://github.com/velocitatem/disgraPhi/blob/main/apps/webapp-minimal/app.py) to collect high-quality samples with guided annotation. Run it locally, then upload the exported dataset here!

## Steps

1. Setup environment
2. Upload your handwriting samples
3. Create ground truth labels
4. Train personalized model
5. Test your model
6. Download trained model

Let's get started! 🚀

## Step 1: Setup Environment

First, let's install DisgraPhi and its dependencies. This will take 2-3 minutes.

In [ ]:
# Check GPU availability
import subprocess
import sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print("✓ GPU detected:")
    print(gpu_info.stdout.split('\n')[8])  # GPU info line
else:
    print("⚠️  No GPU detected. Training will be very slow.")
    print("   Go to Runtime → Change runtime type → Select 'T4 GPU'")
    sys.exit(1)

In [ ]:
# Clone DisgraPhi repository
!git clone https://github.com/velocitatem/disgraPhi.git
%cd disgraPhi

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate peft bitsandbytes pillow opencv-python numpy tqdm tensorboard
!pip install -q qwen-vl-utils  # For Qwen models

print("\n✓ Installation complete!")

## Step 2: Upload Your Handwriting Samples

Upload images of your handwriting. These can be:
- Photos taken with your phone (JPG/PNG)
- Scanned documents
- Individual lines or full pages

**For best results, use the Gradio Data Collector:**

Instead of manually uploading images, you can use our interactive Gradio webapp locally:
1. Clone the repo and run `python apps/webapp-minimal/app.py`
2. It will guide you through photographing 10 practice sentences
3. Automatically crops and annotates your handwriting
4. Exports a ready-to-use ZIP file
5. Upload the extracted images here

**Or upload directly:**

**Tips for best results:**
- Use clear, well-lit photos
- Include variety (different words, sentences)
- 10-30 samples is ideal (more is better!)
- Each image should be a single line of text

In [ ]:
import os
from pathlib import Path

# Create directories for user data
user_data_dir = Path("/content/disgraPhi/my_handwriting")
user_data_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Created directory: {user_data_dir}")
print("\nNow upload your handwriting images in the next cell.")

In [ ]:
from google.colab import files
import shutil

print("📤 Upload your handwriting images (PNG, JPG, JPEG)")
print("   You can select multiple files at once.\n")

uploaded = files.upload()

# Move uploaded files to data directory
for filename in uploaded.keys():
    src = f"/content/{filename}"
    dst = user_data_dir / filename
    shutil.move(src, dst)
    print(f"✓ Saved: {filename}")

# Count images
image_files = list(user_data_dir.glob("*.png")) + list(user_data_dir.glob("*.jpg")) + list(user_data_dir.glob("*.jpeg"))
print(f"\n✓ Total images uploaded: {len(image_files)}")

## Step 3: Create Ground Truth Labels

Now we need to tell the model what each image says. We'll create a simple interactive labeling interface.

In [ ]:
import json
from IPython.display import display, Image as IPImage, clear_output
from PIL import Image

# Sort images for consistent ordering
image_files = sorted(image_files)

print("📝 Label Your Handwriting Samples")
print("="*50)
print("For each image, type exactly what the text says.")
print("Press Enter to save and move to the next image.\n")

manifest = {
    "created": "2025-10-30T00:00:00Z",
    "totalSamples": len(image_files),
    "samples": []
}

for idx, img_path in enumerate(image_files):
    clear_output(wait=True)
    
    # Display progress
    print(f"\nImage {idx + 1} of {len(image_files)}")
    print("="*50)
    
    # Show image
    img = Image.open(img_path)
    # Resize for display (max width 800px)
    max_width = 800
    if img.width > max_width:
        ratio = max_width / img.width
        new_size = (max_width, int(img.height * ratio))
        img = img.resize(new_size)
    display(img)
    
    # Get ground truth
    text = input(f"\nWhat does this say? ")
    
    # Rename file to standard format
    new_filename = f"sample_{idx:04d}.png"
    new_path = user_data_dir / new_filename
    
    # Convert to PNG if needed and save
    img_orig = Image.open(img_path).convert('RGB')
    img_orig.save(new_path, 'PNG')
    
    # Remove old file if different
    if img_path != new_path:
        img_path.unlink()
    
    manifest["samples"].append({
        "imageFile": new_filename,
        "groundTruth": text,
        "index": idx
    })

# Save manifest
manifest_path = user_data_dir / "manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

clear_output(wait=True)
print("\n✓ Labeling complete!")
print(f"✓ Saved {len(manifest['samples'])} labeled samples")
print(f"✓ Manifest saved to: {manifest_path}")

### Preview Your Dataset

In [ ]:
# Show a few samples
print("📋 Your Labeled Dataset (showing first 3):")
print("="*60)

for i, sample in enumerate(manifest["samples"][:3]):
    print(f"\nSample {i+1}:")
    print(f"  File: {sample['imageFile']}")
    print(f"  Text: {sample['groundTruth']}")

print(f"\n... and {len(manifest['samples']) - 3} more samples")

## Step 4: Train Your Personalized Model

Now for the magic! We'll fine-tune a vision-language model on your handwriting.

**Training settings:**
- Model: SmolVLM-256M (efficient, good for Colab)
- Method: LoRA (parameter-efficient fine-tuning)
- Time: ~15-30 minutes depending on sample count
- Data split: 80% train, 10% validation, 10% test

In [ ]:
# Training configuration
model_name = "smolvlm-256m"  # Efficient model for Colab
num_epochs = 5  # Adjust based on your sample count
batch_size = 2  # Small batch for Colab GPU memory
learning_rate = 2e-5

# Adjust epochs based on dataset size
if len(manifest['samples']) < 15:
    num_epochs = 10  # More epochs for smaller datasets
    print(f"⚠️  Small dataset detected. Using {num_epochs} epochs.")
elif len(manifest['samples']) > 40:
    num_epochs = 3  # Fewer epochs for larger datasets
    print(f"✓ Large dataset detected. Using {num_epochs} epochs.")

print(f"\n🎯 Training Configuration:")
print(f"  Model: {model_name}")
print(f"  Samples: {len(manifest['samples'])}")
print(f"  Epochs: {num_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {learning_rate}")
print(f"\nStarting training...\n")

In [ ]:
# Run training
!python ml/models/train.py \
    --model_provider {model_name} \
    --dataset_type manifest \
    --manifest_data_dir "{user_data_dir}" \
    --num_train_epochs {num_epochs} \
    --per_device_train_batch_size {batch_size} \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate {learning_rate} \
    --output_dir "/content/models" \
    --experiment_name "my_handwriting_personalized" \
    --logging_steps 5 \
    --eval_steps 20 \
    --save_steps 50 \
    --bf16 \
    --augment \
    --augment_strength 0.5

print("\n" + "="*60)
print("✓ Training complete!")
print("="*60)

### Monitor Training Progress (Optional)

View training metrics in TensorBoard:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/models/logs

## Step 5: Test Your Model

Let's see how well your personalized model works!

In [ ]:
import torch
from ml.models.providers import create_model
from PIL import Image

# Load trained model
print("Loading your personalized model...")

adapter_path = "/content/models/my_handwriting_personalized/final_adapter"
model = create_model(model_name)
model.load_adapter(adapter_path)
model.eval()

print("✓ Model loaded!\n")

def transcribe_image(image_path):
    """Transcribe handwriting from image."""
    img = Image.open(image_path).convert('RGB')
    
    with torch.no_grad():
        text = model.generate(
            pixel_values=img,
            prompt="Transcribe this handwritten text.",
            max_new_tokens=128,
            temperature=0.0
        )
    
    return text

In [ ]:
# Test on a few random samples
import random

test_samples = random.sample(manifest["samples"], min(3, len(manifest["samples"])))

print("🧪 Testing on random samples:")
print("="*60)

for sample in test_samples:
    img_path = user_data_dir / sample["imageFile"]
    ground_truth = sample["groundTruth"]
    
    # Show image
    img = Image.open(img_path)
    display(img.resize((min(img.width, 600), int(img.height * min(1.0, 600/img.width)))))
    
    # Transcribe
    prediction = transcribe_image(img_path)
    
    print(f"Ground Truth: {ground_truth}")
    print(f"Prediction:   {prediction}")
    print(f"Match: {'✓ Perfect!' if prediction.strip().lower() == ground_truth.strip().lower() else '~ Close' if prediction.strip()[:10].lower() == ground_truth.strip()[:10].lower() else '✗ Different'}")
    print("\n" + "="*60 + "\n")

### Test on New Handwriting

Upload a new image of your handwriting to test:

In [ ]:
print("📤 Upload a new handwriting sample to test:")

test_uploaded = files.upload()

for filename in test_uploaded.keys():
    test_path = f"/content/{filename}"
    
    # Display image
    img = Image.open(test_path)
    display(img.resize((min(img.width, 600), int(img.height * min(1.0, 600/img.width)))))
    
    # Transcribe
    result = transcribe_image(test_path)
    
    print(f"\n📝 Transcription: {result}")
    print("\n" + "="*60)

## Step 6: Download Your Model

Save your personalized model to use later!

In [ ]:
# Create a package with model and instructions
import zipfile
import json
from datetime import datetime

package_path = "/content/my_handwriting_model.zip"

print("📦 Packaging your model...")

with zipfile.ZipFile(package_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add adapter files
    adapter_dir = Path("/content/models/my_handwriting_personalized/final_adapter")
    for file in adapter_dir.rglob('*'):
        if file.is_file():
            arcname = f"adapter/{file.relative_to(adapter_dir)}"
            zipf.write(file, arcname)
    
    # Add model info
    model_info = {
        "model_name": model_name,
        "training_date": datetime.now().isoformat(),
        "num_samples": len(manifest["samples"]),
        "num_epochs": num_epochs,
        "usage": "Load with: model.load_adapter('adapter')"
    }
    zipf.writestr("model_info.json", json.dumps(model_info, indent=2))
    
    # Add README
    readme = f"""# Your Personalized DisgraPhi Model

Trained on: {datetime.now().strftime('%Y-%m-%d %H:%M')}
Base model: {model_name}
Training samples: {len(manifest['samples'])}
Epochs: {num_epochs}

## How to Use

```python
from ml.models.providers import create_model

model = create_model("{model_name}")
model.load_adapter("adapter")

# Transcribe image
text = model.generate(
    pixel_values=your_image,
    prompt="Transcribe this handwritten text.",
    max_new_tokens=128
)
```
"""
    zipf.writestr("README.md", readme)

print(f"\n✓ Model packaged: {package_path}")
print(f"  Size: {os.path.getsize(package_path) / 1024 / 1024:.1f} MB")

In [ ]:
# Download the package
from google.colab import files

print("📥 Downloading your model package...")
files.download(package_path)

print("\n✓ Download complete!")
print("\nYou can now use this model locally with DisgraPhi.")

## 🎉 Congratulations!

You've successfully created a personalized handwriting recognition model!

### Next Steps

1. **Improve accuracy**: Add more samples and retrain
2. **Use locally**: Set up DisgraPhi on your computer
3. **Deploy**: Use the inference API for real-time transcription

### Tips for Better Results

- Include diverse samples (different words, styles)
- Use clear, well-lit photos
- Label accurately (typos in ground truth affect training)
- More samples = better accuracy (aim for 30+)

### Get Help

- [GitHub Repository](https://github.com/velocitatem/disgraPhi)
- [Documentation](https://github.com/velocitatem/disgraPhi#readme)
- [Open an Issue](https://github.com/velocitatem/disgraPhi/issues)

### Share Your Results

If this helped you, consider:
- ⭐ Starring the repository
- 📢 Sharing with others who might benefit
- 🐛 Reporting bugs or suggesting improvements

Thank you for using DisgraPhi! 🖊️